In [28]:
# --- imports ---
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score

from building_models import baseline_predict, make_pipeline

print("All imports OK")


All imports OK


In [29]:
# --- global var ---
DB = "../sql/TTC_delay.db"

CATEGORICAL = ["transit_mode", "delay_reason", "line", "bound", "day_type"]
NUMERIC = ["hour", "day_num"]
FEATURES = CATEGORICAL + NUMERIC

In [30]:
conn = sqlite3.connect(DB)
df = pd.read_sql('''
    SELECT 
        transit_mode, 
        date, 
        delay_reason, 
        line, 
        bound, 
        day_type,
        hour, 
        day_num, 
        min_delay
    FROM view_fact_delays''', conn)
conn.close()

# prepare data
df["date"] = pd.to_datetime(df["date"])
print(f"Total incidents: {len(df):,}")
print(f"Percentage of dataset with 0 min delay: {(df['min_delay'] == 0).mean():.1%}")

d = df[df["min_delay"] > 0].copy()
print(f"Modelled incidents: {len(d):,}")
print(d["min_delay"].quantile([0.5, 0.9, 0.99, 1.0]).rename("minutes"))

display(df.info())

Total incidents: 171,435
Percentage of dataset with 0 min delay: 28.0%
Modelled incidents: 123,513
0.50     11.0
0.90     30.0
0.99    235.0
1.00    999.0
Name: minutes, dtype: float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171435 entries, 0 to 171434
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   transit_mode  171435 non-null  object        
 1   date          171435 non-null  datetime64[ns]
 2   delay_reason  171435 non-null  object        
 3   line          171435 non-null  object        
 4   bound         171435 non-null  object        
 5   day_type      171435 non-null  object        
 6   hour          171435 non-null  int64         
 7   day_num       171435 non-null  int64         
 8   min_delay     171435 non-null  int64         
dtypes: datetime64[ns](1), int64(3), object(5)
memory usage: 11.8+ MB


None

In [31]:
# chronological split: first 80% for training, last 20% for testing
cutoff = d["date"].quantile(0.8)
train = d[d["date"] <= cutoff]
test = d[d["date"] > cutoff]
print(f"train: {train['date'].min().date()} -> {train['date'].max().date()} ({len(train):,} rows)")
print(f"test: {test['date'].min().date()} -> {test['date'].max().date()} ({len(test):,} rows)")

X_train, X_test = train[FEATURES], test[FEATURES]
# log tranforming for heavily skewed dataset
# np.log1p handles delay = 0 cases safely
y_train_log, y_test_log = np.log1p(train["min_delay"]), np.log1p(test["min_delay"])
y_test = test["min_delay"]

train: 2025-01-01 -> 2026-04-19 (98,936 rows)
test: 2026-04-20 -> 2026-08-31 (24,577 rows)


In [32]:
base_pred = baseline_predict(train, test)
print(f"Baseline MAE: {mean_absolute_error(y_test, base_pred):.2f} min")

Baseline MAE: 12.65 min


In [33]:
models = {
    "Ridge Regression":  make_pipeline(Ridge(alpha=1.0), CATEGORICAL, NUMERIC), # linear
    "Gradient Boosting": make_pipeline(HistGradientBoostingRegressor(max_iter=200, 
                                                                    learning_rate=0.1, 
                                                                    random_state=42),
                                        CATEGORICAL, NUMERIC), # ensemble
}

preds = {
    "Baseline: overall median": pd.Series(train["min_delay"].median(), index=test.index),
    "Baseline (median by mode + cause)": base_pred,
}
for name, m in models.items():
    m.fit(X_train, y_train_log)
    # undo log1p with expm1
    preds[name] = pd.Series(np.expm1(m.predict(X_test)).clip(min=0), index=test.index)

/Users/tranphuongvy/Projects/price-optimization/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/tranphuongvy/Projects/price-optimization/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/tranphuongvy/Projects/price-optimization/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/tranphuongvy/Projects/price-optimization/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tranphuongvy/Projects/price-optimization/.venv/lib/python3.9/site-packages/sklearn/linear_model/_base.py:279: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Users/tranphuongvy/Projects/price-optimization/.venv/lib/pyth

In [34]:
# results table
rows = []
for name, p in preds.items():
    rows.append({"model": name,
                 "MAE (min)": mean_absolute_error(y_test, p),
                 "median abs error (min)": median_absolute_error(y_test, p),
                 "R2 (log scale)": r2_score(y_test_log, np.log1p(p))})
results = pd.DataFrame(rows).set_index("model")

base_mae  = results.loc["mode + cause median", "MAE (min)"] # strict baseline
naive_mae = results.loc["overall median", "MAE (min)"] # simple baseline
results["vs. overall median"] = (1 - results["MAE (min)"] / naive_mae).map("{:.1%}".format)
results["vs. mode + cause median"]  = (1 - results["MAE (min)"] / base_mae).map("{:.1%}".format)
results.round(2)

KeyError: 'mode + cause median'

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
results["MAE (min)"].plot.barh(ax=ax)
ax.set_xlabel("Mean absolute error (minutes), lower is better")
ax.set_title("Prediction error on the most recent 20% of incidents")

plt.show()

In [ ]:
best_name = results.loc[list(models), "MAE (min)"].idxmin()
best = models[best_name]
print("model used:", best_name)

# feature importance
imp = permutation_importance(best, X_test, y_test_log,
                            scoring="neg_mean_absolute_error",
                            n_repeats=5, 
                            random_state=42, 
                            n_jobs=-1)

importance = (pd.Series(imp.importances_mean, index=FEATURES).clip(lower=0).sort_values(ascending=False))
importance_pct = (importance / importance.sum()).rename("Percentage of total importance")
importance_pct.map("{:.1%}".format)

In [ ]:
ax = importance_pct.sort_values().plot.barh(figsize=(7, 3.5))
ax.set_xlabel("Percentage of total importance")
ax.set_title(f"What drives delay duration? ({best_name})")

plt.show()